# Content-Based Filtering using TF-IDF

This notebook implements a content-based filtering approach using TF-IDF vectorization for the educational content recommendation system.

## Objectives
1. Process content metadata to extract features
2. Implement TF-IDF vectorization
3. Calculate content similarity
4. Build a content-based recommendation system
5. Evaluate the performance of the system

In [3]:
# Import required libraries
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set up system path
sys.path.append(os.path.abspath(".."))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.utils.preprocessing import preprocess_content_text

# For reproducibility
np.random.seed(42)

# Configure Seaborn style and color palette
sns.set_theme(style="whitegrid", palette="viridis")

## 1. Load and Prepare Data

In [4]:
# Load the dataset
lectures_data = pd.read_csv('../data/cleaned/cleaned_lectures.csv')
merged_data = pd.read_csv('../data/cleaned/merged_cleaned_data.csv')

# Display basic info about the datasets
print(f"Lectures dataset shape: {lectures_data.shape}")
print(f"Merged data shape: {merged_data.shape}")

# Display the first few rows of each dataset
print("\nSample of lectures data:")
lectures_data.head()

Lectures dataset shape: (1021, 6)
Merged data shape: (117167, 14)

Sample of lectures data:


,lecture_id,part,tags,video_length,deployed_at,video_minutes
0,l520,5.0,142.0,NaN,NaN,NaN
1,l592,6.0,142.0,NaN,NaN,NaN
2,l1259,1.0,222.0,359000.0,2019-10-07 05:05:29.123,5.983333
3,l1260,1.0,220.0,487000.0,2019-10-07 05:05:38.105,8.116667
4,l1261,1.0,221.0,441000.0,2019-10-07 05:05:43.162,7.350000


In [5]:
# Display merged data
print("\nSample of merged data:")
merged_data.head()


Sample of merged data:


,timestamp,solving_id,question_id,user_answer,elapsed_time,user_id,bundle_id,explanation_id,correct_answer,part,tags,deployed_at,is_correct,tag_difficulty
0,2018-03-25 09:57:07.112,1,q8098,b,12000,u145242,b5569,e5569,b,1,5;2;182,2017-12-29 15:06:23.093,True,0.665909
1,2018-03-25 09:57:35.061,2,q8074,d,25000,u145242,b5545,e5545,c,1,11;7;183,2018-05-18 08:57:02.552,False,0.628323
2,2018-03-25 09:57:59.724,3,q176,b,22000,u145242,b176,e176,d,1,6;7;183,2017-12-29 14:53:43.800,False,0.641848
3,2018-03-25 09:58:19.710,4,q1279,c,17000,u145242,b1279,e1279,c,2,24;26;182;184,2019-10-17 02:58:38.714,True,0.681122
4,2018-03-25 09:59:03.593,5,q2067,a,13333,u145242,b1623,e1623,b,3,52;183;184,2019-03-12 02:28:10.338,False,0.641035


## 2. Process Content Metadata

First, I need to process the content metadata to create features for the TF-IDF model. For the educational content, I'll focus on features like part (section), tags and any other available attributes.

In [6]:
# Create subject categories based on ranges of tag values
# These mappings are based on TOEIC structure
def get_subject_category(tag):
    try:
        tag_val = float(tag)
        if 1 <= tag_val < 23:
            return "Listening Skills"
        elif 23 <= tag_val < 52:
            return "Reading Skills"
        elif 52 <= tag_val < 70:
            return "Speaking Skills"
        elif 70 <= tag_val < 150:
            return "Writing Skills"
        elif 150 <= tag_val < 200:
            return "Test Preparation"
        elif 200 <= tag_val < 300:
            return "Grammar & Vocabulary"
        else:
            return "General"
    except:
        return "General"

lectures_data['subject_category'] = lectures_data['tags'].apply(get_subject_category)

# Map part numbers to human-readable names
part_names = {
    0: "Introduction",
    1: "Listening Comprehension",
    2: "Reading Comprehension",
    3: "Grammar & Vocabulary",
    4: "Speaking Assessment",
    5: "Writing Exercises",
    6: "Practice Tests",
    7: "Additional Resources"
}

lectures_data['part_name'] = lectures_data['part'].map(part_names)

# Display the processed lectures data
print("Processed lectures data:")
lectures_data[['lecture_id', 'part', 'part_name', 'tags', 'subject_category', 'video_length']].head()

Processed lectures data:


,lecture_id,part,part_name,tags,subject_category,video_length
0,l520,5.0,Writing Exercises,142.0,Writing Skills,NaN
1,l592,6.0,Practice Tests,142.0,Writing Skills,NaN
2,l1259,1.0,Listening Comprehension,222.0,Grammar & Vocabulary,359000.0
3,l1260,1.0,Listening Comprehension,220.0,Grammar & Vocabulary,487000.0
4,l1261,1.0,Listening Comprehension,221.0,Grammar & Vocabulary,441000.0


Shows the processed `lectures_data` with new columns `part_name` and `subject_category`, enhancing interpretability.

## 3. Extract Bundle Information

The merged_data contains bundle information which I'll use as my content items for recommendations. Let's extract unique bundles and their associated metadata. 

**Key Operations**:
- Groups `merged_data` by `bundle_id` to extract unique bundles and their metadata (`part`, `tags`).
- Computes `question_count` (number of questions per bundle), `interaction_count` (number of user interactions), and `success_rate` (proportion of correct answers).
- Calculates `bundle_difficulty` as the mean `tag_difficulty` per bundle.
- Maps `part` to `part_name` and `tags` to `subject_category` using previously defined functions/dictionaries.
- Prints the total number of bundles and displays the first five rows of the processed `bundle_features` DataFrame.

In [7]:
# Extract unique bundle information
bundle_info = merged_data.groupby('bundle_id').agg({
    'part': 'first',
    'tags': lambda x: ';'.join(set(str(i) for i in x if pd.notna(i))),
    'question_id': lambda x: len(set(x))  # Number of questions in bundle
}).reset_index()

# Rename columns for clarity
bundle_info.columns = ['bundle_id', 'part', 'tags', 'question_count']

# Map part to human-readable names
bundle_info['part_name'] = bundle_info['part'].map(part_names)

# Create subject category based on tags
bundle_info['subject_category'] = bundle_info['tags'].apply(get_subject_category)

# Calculate bundle popularity from interaction data
bundle_popularity = merged_data['bundle_id'].value_counts().reset_index()
bundle_popularity.columns = ['bundle_id', 'interaction_count']

# Calculate bundle difficulty from correct answer rates
bundle_difficulty = merged_data.groupby('bundle_id').apply(
    lambda x: (x['user_answer'] == x['correct_answer']).mean()
).reset_index()
bundle_difficulty.columns = ['bundle_id', 'success_rate']

# Merge all features
bundle_features = bundle_info.merge(bundle_popularity, on='bundle_id', how='left')
bundle_features = bundle_features.merge(bundle_difficulty, on='bundle_id', how='left')

# Fill missing values
bundle_features['interaction_count'] = bundle_features['interaction_count'].fillna(0)
bundle_features['success_rate'] = bundle_features['success_rate'].fillna(0.5)

# Display the bundle features
print(f"Total bundles: {len(bundle_features)}")
bundle_features.head()

Total bundles: 8260


C:\Users\karat\AppData\Local\Temp\ipykernel_15484\3572219143.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bundle_difficulty = merged_data.groupby('bundle_id').apply(


,bundle_id,part,tags,question_count,part_name,subject_category,interaction_count,success_rate
0,b1,1,1;2;179;181,1,Listening Comprehension,General,6,0.833333
1,b10,1,17;7;182,1,Listening Comprehension,General,47,0.319149
2,b100,1,22;2;181,1,Listening Comprehension,General,7,1.000000
3,b1000,2,24;33;182;183,1,Reading Comprehension,General,46,0.760870
4,b1001,2,34;35;182;183,1,Reading Comprehension,General,12,0.583333


## 4. Implement TF-IDF Vectorization

Now, I'll create a text representation for each bundle, process it and then apply TF-IDF vectorization to convert them into numerical feature vectors.

In [8]:
# Create a text representation for each bundle
bundle_features['content_text'] = (
    bundle_features['part_name'].fillna('') + ' ' +
    bundle_features['subject_category'].fillna('') + ' ' +
    bundle_features['tags'].fillna('')
)

# Display the text representation for a few bundles
print("Text representation for TF-IDF:")
bundle_features[['bundle_id', 'content_text']].head()

Text representation for TF-IDF:


,bundle_id,content_text
0,b1,Listening Comprehension General 1;2;179;181
1,b10,Listening Comprehension General 17;7;182
2,b100,Listening Comprehension General 22;2;181
3,b1000,Reading Comprehension General 24;33;182;183
4,b1001,Reading Comprehension General 34;35;182;183


In [9]:
# Apply preprocessing
bundle_features['content_text'] = bundle_features['content_text'].apply(preprocess_content_text)
bundle_features[['bundle_id', 'content_text']].head()

,bundle_id,content_text
0,b1,listening comprehension general 1 2 179 181
1,b10,listening comprehension general 17 7 182
2,b100,listening comprehension general 22 2 181
3,b1000,reading comprehension general 24 33 182 183
4,b1001,reading comprehension general 34 35 182 183


In [10]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # Limit the number of features to avoid dimensionality issues
    stop_words='english',
    ngram_range=(1, 2)  # Include both unigrams and bigrams
)

# Fit and transform the content text
tfidf_matrix = tfidf_vectorizer.fit_transform(bundle_features['content_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# Show the feature names (terms)
print("\nSample of TF-IDF feature names:")
feature_names = tfidf_vectorizer.get_feature_names_out()
print(feature_names[:10], '...')

TF-IDF matrix shape: (8260, 1459)

Sample of TF-IDF feature names:
['10' '10 179' '10 181' '10 182' '10 183' '10 184' '10 185' '100' '100 76'
 '100 85'] ...


In [11]:
# Define parameter grid for TF-IDF tuning
tfidf_params_grid = [
    {'max_features': 5000, 'stop_words': 'english', 'ngram_range': (1, 2), 'min_df': 0.01, 'max_df': 0.95},
    {'max_features': 7000, 'stop_words': 'english', 'ngram_range': (1, 3), 'min_df': 0.01, 'max_df': 0.95},
    {'max_features': 10000, 'stop_words': 'english', 'ngram_range': (1, 2), 'min_df': 0.01, 'max_df': 0.85}
]

# Define helper functions for recommendation & ground truth
def get_users_for_eval():
    user_counts = merged_data['user_id'].value_counts()
    return user_counts[user_counts >= 10].sample(20, random_state=42).index.tolist()

def build_recommend_fn(tfidf_matrix, vectorizer, min_similarity=0.1):
    sim = cosine_similarity(tfidf_matrix)
    indices = pd.Series(bundle_features.index, index=bundle_features['bundle_id'])
    
    def recommend_fn(user_id, k=10):
        user_history = merged_data[merged_data['user_id'] == user_id]
        seen_bundles = set(user_history['bundle_id'].unique())
        rec_scores = {}
        
        for b in seen_bundles:
            if b not in indices: continue
            idx = indices[b]
            for i, score in enumerate(sim[idx]):
                b_id = bundle_features.iloc[i]['bundle_id']
                if b_id not in seen_bundles and score >= min_similarity:
                    rec_scores[b_id] = rec_scores.get(b_id, 0) + score
            
        # Normalize scores
        max_score = max(rec_scores.values()) if rec_scores else 1
        rec_scores = {k: v/max_score for k, v in rec_scores.items()}
        
        # Get top recommendations
        top_bundles = sorted(rec_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        return [b[0] for b in top_bundles]
    
    return recommend_fn

def get_relevant_fn(user_id):
    return merged_data[merged_data['user_id'] == user_id]['bundle_id'].unique().tolist()

## 5. Calculate Content Similarity

Next, I'll calculate the cosine similarity between all pairs of content items (bundles) to identify items that are similar to each other.

In [12]:
# Calculate cosine similarity between all bundles
cosine_sim = cosine_similarity(tfidf_matrix)
print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

# Create a mapping from bundle IDs to matrix indices
bundle_indices = pd.Series(bundle_features.index, index=bundle_features['bundle_id'])

# Example: Show similarity scores for a sample bundle
sample_bundle_id = bundle_features['bundle_id'].iloc[10]  # Get a sample bundle ID
sample_idx = bundle_indices[sample_bundle_id]

print(f"\nSimilarity scores for bundle {sample_bundle_id}:")
# Get pairwise similarity scores and sort them
sim_scores = list(enumerate(cosine_sim[sample_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

# Print the top 5 most similar bundles (excluding itself)
top_similar = sim_scores[1:6]  # Skip the first one (itself)
for i, score in top_similar:
    similar_bundle_id = bundle_features.iloc[i]['bundle_id']
    print(f"Bundle {similar_bundle_id}: Similarity = {score:.4f}")

Cosine similarity matrix shape: (8260, 8260)

Similarity scores for bundle b1007:
Bundle b12088: Similarity = 1.0000
Bundle b12089: Similarity = 1.0000
Bundle b1272: Similarity = 1.0000
Bundle b1357: Similarity = 1.0000
Bundle b311: Similarity = 1.0000
